# 🚀 Vertex AI & vLLM 기반 Gemma 4 12B 파인튜닝 및 커스텀 컨테이너 배포 실전 파이프라인

본 jupyter notebook은 **Gemma 4 12B-it** 모델을 사내 지식 기반으로 Fine-tuning(미세조정)하고, 이를 초고속 추론 엔진인 **vLLM**과 결합하여 **Google Vertex AI Endpoint(L4 GPU 가속)**에 배포하는 전체 엔드투엔드(End-to-End) 실전 가이드입니다.

### 💡 핵심 아키텍처 흐름
1. **환경 초기화 및 GCP API 활성화**
2. **순정 Base Model 배포 및 추론 (Model Garden API)**
3. **Supervised Fine-Tuning (SFT QLoRA 학습)**
4. **가중치 내장형(Bake-in) 커스텀 vLLM 컨테이너 이미지 빌드**
5. **Artifact Registry에 이미지 업로드 및 Vertex AI 배포**
6. **파인튜닝 완료 모델 최종 예측 검증 (OpenAI 호환 규격)**

---

## 📦 1. 공통 환경 준비 (GCP 초기화 및 SDK 설치)

발표와 실습을 위해 필요한 라이브러리(SDK)를 설치하고 GCP 서비스(Vertex AI, Artifact Registry) API를 활성화한 뒤, Project ID와 Region을 연동합니다.

In [ ]:
# 학습 및 배포용 필수 패키지 설치
!pip install -q --upgrade google-cloud-aiplatform datasets transformers trl peft bitsandbytes accelerate openai google-cloud-artifact-registry google-cloud-service-usage

In [ ]:
# 1. SDK ServiceUsage API를 활용하여 Artifact Registry API 활성화
from google.cloud import service_usage_v1

# 💡 [노트북 핵심 설정 변수 취합]
# 노트북 전체에서 공유하여 사용하는 모든 핵심 매개변수들을 중앙 제어식으로 일괄 선언합니다.
PROJECT_ID = "<your-project-id>"                                        # GCP Project ID (공통)
LOCATION = "asia-southeast1"                                        # GCP 리전 (L4 GPU 가용 싱가포르)
REGION = "asia-southeast1"                                          # Artifact Registry 생성용 리전 명칭
MACHINE_TYPE = "g4-standard-48"
ACCELERATOR_TYPE = "NVIDIA_RTX_PRO_6000"
ACCELERATOR_COUNT = 1
BUCKET_NAME="<your bucket name>"

# [2번 섹션] 순정 Base Model 배포용 변수
MODEL_GARDEN_GEMMA4_ID = "gemma-4-12b-it"                           # Model Garden 타겟 모델 ID
BASE_MODEL_DISPLAY_NAME = f"gemma4-base-deploy"
BASE_ENDPOINT_DISPLAY_NAME = f"gemma4-base-endpoint"

# [4번 섹션] SFT 파인튜닝용 변수
MODEL_ID = "google/gemma-4-12b-it"                                  # HuggingFace 로드용 원본 모델 ID
TRAIN_DATA_PATH = "gs://{BUCKET_NAME}/bigquery_faq_finetune.jsonl"  # GCS 내 파인튜닝 학습 데이터셋 경로
OUTPUT_DIR = "gs://{BUCKET_NAME}/finetuned_models/gemma4-hf-lora"    # 학습 가중치 백업용 GCS 버킷 경로

# [5번 섹션] 커스텀 컨테이너 이미지 및 Artifact Registry 빌드용 변수
REPOSITORY_ID = "gemma4-vllm-repo"                                  # Artifact Registry 신규 도커 저장소 ID
CONTAINER_NAME = "gemma4-vllm"                                      # 커스텀 도커 이미지 이름
CONTAINER_TAG = "v1"                                                # 커스텀 도커 이미지 태그

# 💡 상위 REGION, PROJECT_ID, REPOSITORY_ID, CONTAINER_NAME, CONTAINER_TAG 변수를 동적 조합하여 최종 URI 생성
CUSTOM_VLLM_CONTAINER = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{REPOSITORY_ID}/{CONTAINER_NAME}:{CONTAINER_TAG}"

# [6번 섹션] Vertex AI Model Registry 및 배포용 변수
MODEL_NAME = "gemma4-12b-it-lora-serve"                             # Model Registry에 등록할 신규 모델 Display Name

# ServiceUsage 클라이언트 초기화 및 API 활성화
client = service_usage_v1.ServiceUsageClient()
service_name = f"projects/{PROJECT_ID}/services/artifactregistry.googleapis.com"
request = service_usage_v1.EnableServiceRequest(name=service_name)

print(f"[{PROJECT_ID}] 프로젝트에서 Artifact Registry API 활성화를 시작합니다...")
operation = client.enable_service(request=request)
operation.result()  # 대기
print("🎉 Artifact Registry API 활성화 완료 및 변수 동적 매핑 등록 완료!")

In [ ]:
import vertexai
from google.cloud import aiplatform
import os

# 전역 설정된 PROJECT_ID 및 LOCATION을 사용하여 SDK 초기화
vertexai.init(project=PROJECT_ID, location=LOCATION)
aiplatform.init(project=PROJECT_ID, location=LOCATION)

print(f"✅ GCP SDK 연동 완료! (Project: {PROJECT_ID}, Location: {LOCATION})")

## 🎯 2. 순정 Base Model 배포 (Model Garden API)

학습을 진행하기 전, 구글 공식 Model Garden 저장소로부터 순정 Gemma 4 12B IT 모델을 읽어와서 Endpoint에 배포합니다.

In [ ]:
from vertexai import model_garden
import time

try:
    print(f"🔍 Model Garden에서 {MODEL_GARDEN_GEMMA4_ID} 모델 스펙을 로드합니다...")
    model = model_garden.OpenModel(f"publishers/google/models/gemma4@{MODEL_GARDEN_GEMMA4_ID.lower()}")

    print("🚀 Base Model 배포 프로세스를 시작합니다 (배포 완료까지 약 15~20분 소요)...\n")
    base_model_endpoint = model.deploy(
        machine_type=MACHINE_TYPE,                  # L4 GPU 1장 탑재 머신
        accelerator_type=ACCELERATOR_TYPE,
        accelerator_count=ACCELERATOR_COUNT,
        endpoint_display_name=BASE_ENDPOINT_DISPLAY_NAME,
        accept_eula=True                                # 구글 Gemma 라이선스 약관 동의
    )
    print(f"\n✅ Base Model 배포 성공! Endpoint ID: {base_model_endpoint.resource_name}")
except Exception as e:
    print(f"❌ 배포 과정 에러 발생: {e}")

## 🔍 3. Base Model 성능 확인 (추론 테스트)

구글 공식 사전 빌드 이미지는 Vertex AI 표준 `instances` 포맷의 해석기(Middleware)를 내장하고 있습니다. 순정 상태에서 사내 신제품(AuraLight) 정보를 묻는 프롬프트를 전송하여 미학습 상태의 대답을 확인합니다.

In [ ]:
# 구글 공식 가이드에 따른 Vertex AI 표준 예측 포맷 구성
test_prompt = "사내 'Aura-FinOps' 규정에 따른 BigQuery 부서별 비용(Cost Allocation) 분류 시 필수 지정 라벨과 누락 시 제재 사항이 뭐야?"

instances = [
    {
        "@requestFormat": "chatCompletions",
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": test_prompt
                    }
                ]
            }
        ],
        "max_tokens": 2048
    }
]

print("💬 Base Model Endpoint로 질문을 전송합니다...")
response = base_model_endpoint.predict(instances=instances)
print("\n================ [ 순정 모델 답변 결과 ] ================")
print(response.predictions["choices"][0]["message"]["content"])
print("========================================================")

## 🛠️ 4. 사내 데이터 기반 Gemma 4 12B 파인튜닝 (SFT SFTTrainer)

호스트 VM의 GPU를 가동하여 사내 FAQ 데이터셋(`bigquery_faq_finetune.jsonl`)을 읽고, QLoRA 기반 미세 조정을 수행한 후 완성된 LoRA 가중치를 GCS 버킷에 자동 전송합니다.

In [ ]:
import warnings
import logging
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["BITSANDBYTES_NOWELCOME"] = "1"

print("📥 학습 데이터셋을 로드합니다...")
dataset = load_dataset("json", data_files=TRAIN_DATA_PATH, split="train")

# 1. 4비트 QLoRA 양자화 정의
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 2. 모델 및 토크나이저 초기화
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

# 3. LoRA 레이어 매핑
peft_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)

# 4. 학습 매개변수 설정
training_args = SFTConfig(
    output_dir="./results", num_train_epochs=1, max_steps=100,
    per_device_train_batch_size=2, gradient_accumulation_steps=4,
    learning_rate=5e-5, logging_steps=10, bf16=True, optim="paged_adamw_8bit",
    dataset_text_field="text", max_length=4096
)

# 5. Gemma 4 Chat Template 전처리 적용
def formatting_prompts_func(examples):
    output_texts = []
    for i in range(len(examples['input'])):
        formatted_text = f"<|turn>user\n{examples['input'][i]}<turn|>\n<|turn>model\n{examples['output'][i]}<turn|>"
        output_texts.append(formatted_text)
    return {"text": output_texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

# 6. Trainer 실행 및 가중치 업로드
trainer = SFTTrainer(model=model, train_dataset=dataset, peft_config=peft_config, processing_class=tokenizer, args=training_args)
print("🚀 Gemma 4 SFT 파인튜닝 시작...")
trainer.train()

trainer.model.save_pretrained("./final_lora_weights")
tokenizer.save_pretrained("./final_lora_weights")
os.system(f"gcloud storage cp -r ./final_lora_weights/* {OUTPUT_DIR}")
print(f"✅ 파인튜닝 가중치 버킷 업로드 완료: {OUTPUT_DIR}")

## 🐳 5. 가중치 내장형 커스텀 vLLM 컨테이너 빌드 & Artifact Registry 배포

미세조정 완료된 LoRA 어댑터 가중치를 내장형(Bake-in)으로 서빙하기 위해, Artifact Registry 저장소를 신규 개설하고 각 런처 파일과 Dockerfile 빌드 단계를 순차적으로 실행하여 배포를 완료합니다.

### 5-1. Artifact Registry Docker 저장소 생성
Vertex AI 엔드포인트와 근접한 asia-southeast1 리전에 도커 이미지 적재용 Repository를 생성합니다.

In [ ]:
from google.cloud import artifactregistry_v1

client = artifactregistry_v1.ArtifactRegistryClient()
parent = f"projects/{PROJECT_ID}/locations/{REGION}"
repository = artifactregistry_v1.Repository(
    format_=artifactregistry_v1.Repository.Format.DOCKER,
    description="Gemma4 커스텀 vLLM 서빙 이미지를 위한 Docker 저장소"
)

request = artifactregistry_v1.CreateRepositoryRequest(
    parent=parent,
    repository_id=REPOSITORY_ID,
    repository=repository,
)

print(f"[{REGION}] 리전에 [{REPOSITORY_ID}] Artifact Registry 생성을 시작합니다...")
try:
    operation = client.create_repository(request=request)
    response = operation.result()
    print("✅ Artifact Registry 생성 성공!")
    print(f"   - Registry Endpoint: {REGION}-docker.pkg.dev/{PROJECT_ID}/{REPOSITORY_ID}")
except Exception as e:
    print(f"ℹ️ 저장소 상태 확인 (이미 생성되었을 수 있음): {e}")

### 5-2. Google Cloud 가이드 규격 `entrypoint.sh` 런처 스크립트 작성
GCS 버킷 및 Bake-in 로컬 주소 로딩에 모두 대응하는 구글 공식 호환형 서빙 엔트리포인트를 쉘 스크립트 파일로 동적 생성합니다.

In [ ]:
%%writefile entrypoint.sh
#!/bin/bash
# Copyright 2025 Google LLC
# Google 공식 가이드 호환형 vLLM entrypoint 런처 스크립트입니다.

set -euo pipefail

readonly LOCAL_MODEL_DIR=${LOCAL_MODEL_DIR:-"/tmp/model_dir"}

download_model_from_gcs() {
    gcs_uri=$1
    mkdir -p $LOCAL_MODEL_DIR
    echo "Downloading model from $gcs_uri to local directory..."
    if gcloud storage cp -r "$gcs_uri/*" "$LOCAL_MODEL_DIR"; then
      echo "Model downloaded successfully to ${LOCAL_MODEL_DIR}."
    else
      echo "Failed to download model from Cloud Storage: $gcs_uri." >&2
      exit 1
    fi
}

updated_args=()
model_arg="--model="
gcs_protocol="gs://"
for a in "$@"; do
    if [[ $a == $model_arg* ]]; then
        model_path=${a#*=}
        echo $model_path
        if [[ $model_path == $gcs_protocol* ]]; then
            download_model_from_gcs $model_path
            updated_args+=("--model=${LOCAL_MODEL_DIR}")
        else
            # Bake-in된 컨테이너 로컬 경로일 경우 복사 없이 즉시 로드
            updated_args+=("--model=${model_path}")
        fi
    else
        updated_args+=("$a")
    fi
done

echo "Launch command: " "${updated_args[@]}"
exec "${updated_args[@]}"

### 5-3. `Dockerfile.base` (1단계 - vLLM 순정 베이스 빌더) 작성
구글 버텍스 AI 전용 사전 빌드 PyTorch GPU 이미지를 상속하여, gcloud CLI 및 헬퍼 구동 쉘을 안착시키는 순정 베이스 빌더 파일입니다.

In [ ]:
%%writefile Dockerfile.base
# 💡 [1단계] 구글 공식 가이드에 명시된 순정(Vanilla) 베이스 Dockerfile
ARG BASE_IMAGE=vllm/vllm-openai
FROM ${BASE_IMAGE}

ENV DEBIAN_FRONTEND=noninteractive
# Install gcloud SDK
RUN apt-get update && \
    apt-get install -y apt-utils git apt-transport-https gnupg ca-certificates curl \
    && echo "deb [signed-by=/usr/share/keyrings/cloud.google.gpg] https://packages.cloud.google.com/apt cloud-sdk main" | tee -a /etc/apt/sources.list.d/google-cloud-sdk.list \
    && curl https://packages.cloud.google.com/apt/doc/apt-key.gpg | gpg --dearmor -o /usr/share/keyrings/cloud.google.gpg \
    && apt-get update -y && apt-get install google-cloud-cli -y \
    && rm -rf /var/lib/apt/lists/*

WORKDIR /workspace/vllm

# Copy entrypoint.sh to the container
COPY ./entrypoint.sh /workspace/vllm/vertexai/entrypoint.sh
RUN chmod +x /workspace/vllm/vertexai/entrypoint.sh

ENTRYPOINT ["/workspace/vllm/vertexai/entrypoint.sh"]

### 5-4. `Dockerfile` (2단계 - 최종 가중치 Bake-in 빌더) 작성
1단계에서 컴파일 완료된 `gemma4-vllm-base:v1` 순정 이미지를 기반 레이어로 지정한 후, 로컬 디렉토리 내에 보관 중인 **Gemma 4 베이스 모델 가중치**와 **학습 완료된 LoRA 가중치**를 이미지의 로컬 디스크(/workspace) 안으로 한층 더 병합 복사시킵니다.

In [ ]:
%%writefile Dockerfile
# 💡 [2단계] 1단계에서 빌드한 순정 로컬 베이스 이미지를 기반으로 가중치를 Bake-in 시킵니다.
FROM gemma4-vllm-base:v1

# [Bake-in] 로컬 빌드 디렉토리의 가중치(LoRA & Base)를 이미지 내부로 복사
COPY ./models/gemma4-12b-base /workspace/models/gemma4-12b-base
COPY ./final_lora_weights /workspace/lora/gemma4-hf-lora
RUN chmod -R 755 /workspace/models /workspace/lora

### 5-5. 2-Stage Docker Build 및 Artifact Registry 최종 업로드(Push)
로컬 도커 캐시를 가동하여 베이스 이미지와 최종 병합 이미지를 각각 빌드하고, Artifact Registry 저장소 인증 취득 후 클라우드로 안전하게 릴리즈합니다.

In [ ]:
import subprocess
import sys
import os
import shutil
import json

# 변수 재바인딩
WORKBENCH_BASE_MODEL_DIR = "/home/jupyter/gemma4-12b-base"  
WORKBENCH_LORA_DIR = "/home/jupyter/final_lora_weights"      # 파인튜닝 LoRA 최종 출력 경로

BASE_IMAGE_TAG = "gemma4-vllm-base:v1"
FINAL_IMAGE_TAG = "gemma4-vllm:v1"
FULL_IMAGE_URI = CUSTOM_VLLM_CONTAINER

def run_command(command_list):
    """실시간으로 명령어 빌드 로그를 노트북 출력창에 표시하는 헬퍼"""
    process = subprocess.Popen(
        command_list, 
        stdout=subprocess.PIPE, 
        stderr=subprocess.STDOUT, 
        text=True,
        bufsize=1
    )
    for line in process.stdout:
        print(line, end="")
    process.wait()
    if process.returncode != 0:
        raise Exception(f"❌ 명령어 실행 실패 (Exit Code: {process.returncode})")

# ------------------------------------------------------------
# Step 1: Artifact Registry 자격 증명 (Docker 인증)
# ------------------------------------------------------------
print("1단계: Artifact Registry에 Docker 인증을 수행합니다...")
run_command(["gcloud", "auth", "configure-docker", f"{REGION}-docker.pkg.dev", "--quiet"])

# ------------------------------------------------------------
# Step 2: 1단계 - 순정 Dockerfile.base 로컬 빌드
# ------------------------------------------------------------
print(f"\n2단계: 순정 Dockerfile.base 기반으로 로컬 베이스 이미지({BASE_IMAGE_TAG})를 빌드합니다...")
run_command([
    "docker", "build", 
    "-t", BASE_IMAGE_TAG, 
    "-f", "Dockerfile.base", 
    "."
])

# ------------------------------------------------------------
# Step 3: 빌드 컨텍스트에 가중치 파일 사전 복사 및 보정 (중요)
# ------------------------------------------------------------
print("\n3단계: 2단계 빌드를 위해 가중치 파일들을 빌드 디렉토리로 복사/검증합니다...")
dest_base_dir = "models/gemma4-12b-base"
dest_lora_dir = "lora/gemma4-hf-lora"

# 베이스 모델 캐시 검사 및 복사
if os.path.exists(dest_base_dir) and len(os.listdir(dest_base_dir)) > 0:
    print(f"   - ✅ 베이스 모델 가중치 로컬 캐시 확인.")
else:
    if os.path.exists(dest_base_dir): shutil.rmtree(dest_base_dir)
    os.makedirs("models", exist_ok=True)
    if os.path.exists(WORKBENCH_BASE_MODEL_DIR) and len(os.listdir(WORKBENCH_BASE_MODEL_DIR)) > 0:
        print(f"   - 📥 [로컬 복사] {WORKBENCH_BASE_MODEL_DIR} -> {dest_base_dir}")
        shutil.copytree(WORKBENCH_BASE_MODEL_DIR, dest_base_dir)
    else:
        print(f"   - 📥 [GCS 다운로드] gs://... -> models/")
        run_command(["gcloud", "storage", "cp", "-r", f"gs://{BUCKET_NAME}/models/gemma4-12b-base", "models/"])

# LoRA 어댑터 복사
if os.path.exists(dest_lora_dir) and len(os.listdir(dest_lora_dir)) > 0:
    print(f"   - ✅ LoRA 어댑터 가중치 로컬 캐시 확인.")
else:
    if os.path.exists(dest_lora_dir): shutil.rmtree(dest_lora_dir)
    os.makedirs("lora", exist_ok=True)
    if os.path.exists(WORKBENCH_LORA_DIR) and len(os.listdir(WORKBENCH_LORA_DIR)) > 0:
        print(f"   - 📥 [로컬 복사] {WORKBENCH_LORA_DIR} -> {dest_lora_dir}")
        shutil.copytree(WORKBENCH_LORA_DIR, dest_lora_dir)
    else:
        print(f"   - 📥 [GCS 다운로드] gs://... -> lora/")
        run_command(["gcloud", "storage", "cp", "-r", f"gs://{BUCKET_NAME}/final_lora_weights", "lora/gemma4-hf-lora"])

# ------------------------------------------------------------
# Step 4: 2단계 - 베이스 이미지 기반으로 가중치를 담은 최종 이미지 빌드
# ------------------------------------------------------------
print(f"\n4단계: 로컬 베이스 이미지({BASE_IMAGE_TAG})를 상속하여 가중치 복사형 최종 이미지({FULL_IMAGE_URI})를 빌드합니다...")
run_command([
    "docker", "build", 
    "-t", FULL_IMAGE_URI, 
    "-f", "Dockerfile", 
    "."
])

# ------------------------------------------------------------
# Step 5: 최종 이미지를 Artifact Registry로 업로드(Push)
# ------------------------------------------------------------
print(f"\n5단계: 완성된 커스텀 컨테이너 이미지를 Artifact Registry로 업로드(Push)합니다...")
run_command(["docker", "push", FULL_IMAGE_URI])

print(f"\n🎉 모든 작업 완료! 이제 아래의 컨테이너 URI로 배포를 실행합니다:")
print(f"👉 {FULL_IMAGE_URI}")

## 🚀 6. 커스텀 컨테이너 배포 (Vertex AI Model Registry & Endpoint)

업로드된 커스텀 컨테이너 이미지를 Model Registry에 올리고, 예측 엔드포인트에 배포합니다. 

> [!IMPORTANT]
> vLLM의 초기 CUDA Graph 캡처로 인한 배포 타임아웃 판정 오작동을 극복하기 위해 **`--enforce-eager`** 및 **`serving_container_deployment_timeout=3600`** 설정을 적용했습니다.

In [ ]:
import time
from google.cloud import aiplatform

if 'TUNED_MODEL_DISPLAY_NAME' not in globals():
    TUNED_MODEL_DISPLAY_NAME = f"gemma4-tuned-model-{int(time.time())}"
    
model = aiplatform.Model.upload(
    display_name=TUNED_MODEL_DISPLAY_NAME,
    serving_container_image_uri=CUSTOM_VLLM_CONTAINER,
    serving_container_environment_variables={},
    serving_container_command=None,
    # 💡 [Bake-in 핵심] vLLM이 기동될 때 컨테이너 내부 (/workspace/) 로컬 디스크 경로를 직접 바라봅니다.
    serving_container_args=[
        "python3",
        "-m",
        "vllm.entrypoints.openai.api_server",
        "--host=0.0.0.0",
        "--port=8080",
        "--model=/workspace/models/gemma4-12b-base",                # 내장된 베이스 모델 경로
        "--enable-lora",
        "--lora-modules=gemma4-lora=/workspace/lora/gemma4-hf-lora",  # 내장된 LoRA 어댑터 경로
        "--tensor-parallel-size=1",
        "--trust-remote-code",
        "--gpu-memory-utilization=0.85",                             # VRAM 오버헤드 완화 안전 마진 확보
        "--max-model-len=4096",
        "--enforce-eager"                                            # 기동 타임아웃 방지
    ],
    serving_container_ports=[8080],
    serving_container_predict_route="/v1/chat/completions",
    serving_container_health_route="/health",
    serving_container_deployment_timeout=3600,   
)
print(f"✅ Model Registry 등록 성공! Resource Name: {model.resource_name}")

print(f"✅ 모델 등록 완료! (Model ID: {model.resource_name})")

print("\n🚀 2단계: 신규 Endpoint를 개설하고 모델을 인스턴스에 배포합니다...")
endpoint_display_name = f"{MODEL_NAME}-endpoint-{int(time.time())}"
tuned_model_endpoint = aiplatform.Endpoint.create(display_name=endpoint_display_name)

endpoint = model.deploy(
    endpoint=tuned_model_endpoint,
    deployed_model_display_name=MODEL_NAME,
    machine_type=MACHINE_TYPE,                  
    accelerator_type=ACCELERATOR_TYPE,
    accelerator_count=ACCELERATOR_COUNT,
    traffic_percentage=100,
    deploy_request_timeout=3600,
    min_replica_count=1,
    max_replica_count=1,
    service_account=None                                     # ActAs 권한 충돌 방지
)

print(f"\n🎉 배포 최종 성공! Active Endpoint ID: {endpoint.resource_name}")

## 💬 7. 파인튜닝 모델 최종 검증 (OpenAI API 호환 호출)

순정 vLLM 엔드포인트는 OpenAI 표준 JSON 통신을 지원하므로, Vertex AI의 instances 래핑 처리를 바이패스하기 위해 **`raw_predict`** API로 완성도 높은 사내 FAQ 답변 결과를 도출합니다.

In [ ]:
import json

test_prompt = "사내 'Aura-FinOps' 규정에 따른 BigQuery 부서별 비용(Cost Allocation) 분류 시 필수 지정 라벨과 누락 시 제재 사항이 뭐야?"

# vLLM Chat Completions 호환 payload 구성
request_payload = {
    "model": "gemma4-lora",
    "messages": [
        {"role": "user", "content": test_prompt}
    ],
    "max_tokens": 2048,
    "temperature": 0.2,
    "top_p": 0.95
}

print("💬 raw_predict API로 미세 조정된 모델에 질문을 제출합니다...")
raw_response = endpoint.raw_predict(
    body=json.dumps(request_payload),
    headers={"Content-Type": "application/json"}
)

if raw_response.status_code == 200:
    result = json.loads(raw_response.text)
    print("\n================ [ 🎯 파인튜닝 모델 답변 결과 ] ================")
    print(result["choices"][0]["message"]["content"])
    print("===============================================================")
else:
    print(f"❌ 호출 에러 (Status Code: {raw_response.status_code})")
    print(raw_response.text)

## 🛠️ [옵션 1] Workbench VM GPU 로컬 서빙 검증 

Vertex AI Endpoint 배포에 앞서, 주피터 커널이 구동 중인 Workbench VM 내부에서 vLLM 서빙을 테스트할 수 있는 명령어 가이드입니다.

> [!IMPORTANT]
> 실행 전 **Kernel -> Restart Kernel...**을 눌러 학습 세션의 VRAM 점유를 반드시 초기화해 주어야 에러가 나지 않습니다.

In [ ]:
print(f"""👉 [VRAM 22GB 한계 돌파형 로컬 도커 기동 쉘 스크립트]
Jupyter VM 터미널에 아래 구문을 복사하여 실행해 주시면 로컬에서 vLLM 서버가 1분 만에 초고속 기동됩니다:

docker run --gpus all \\
  -p 8000:8080 \\
  --ipc=host \\
  --rm \\
  -e PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \\
  -e VLLM_USE_V1=0 \\                    # V1 이중 프로세스 VRAM 경합 해소
  --name gemma4-vllm-local-serve \\
  {CUSTOM_VLLM_CONTAINER} \\
  python3 -m vllm.entrypoints.openai.api_server \\
  --host 0.0.0.0 \\
  --port 8080 \\
  --model /workspace/models/gemma4-12b-base \\
  --enable-lora \\
  --lora-modules gemma4-lora=/workspace/lora/gemma4-hf-lora \\
  --gpu-memory-utilization 0.40 \\         # 가중치 영역 최대 양보
  --max-model-len 512 \\                  # KV Cache 용량 다이어트
  --kv-cache-dtype fp8 \\                 # 캐시 8비트 정밀도 하향
  --cpu-offload-gb 4 \\                   # 부족한 VRAM을 CPU 메모리로 오프로드
  --enforce-eager                         # CUDA Graph Capturing 생략
""")

## 🧹 [옵션 2] 자원 삭제 및 비용 최적화 (Clean-up)

데모 완료 후 불필요한 과금 발생을 방지하기 위해 생성된 Endpoint, Model Registry, Artifact Registry 리소스를 일괄 영구 소멸시킵니다.

In [ ]:
import os
from google.cloud import aiplatform

print("🧹 1단계: 활성 엔드포인트 일괄 중단 및 삭제 중...")
endpoints = aiplatform.Endpoint.list(order_by="create_time desc")
for ep in endpoints:
    if "gemma4" in ep.display_name:
        print(f"   - Undeploying and deleting endpoint: {ep.display_name}")
        ep.delete(force=True)

print("\n🧹 2단계: Model Registry 내 등록 모델 일괄 삭제 중...")
models = aiplatform.Model.list(order_by="create_time desc")
for m in models:
    if "gemma4" in m.display_name:
        print(f"   - Deleting model from Registry: {m.display_name}")
        m.delete()

print("\n🧹 3단계: Artifact Registry 도커 저장소 완전 삭제 중...")
os.system(f"gcloud artifacts repositories delete {REPOSITORY_ID} --location={REGION} --quiet")

print("\n🎉 데모에 사용된 모든 클라우드 인프라 자원이 안전하게 정리되었습니다!")